   ... binary bomb lab is genuinely one of the most fun coursework exercises in
   computing -- it's like a puzzle boc where each phase teaches you something
   new about how programs actually work at the machine level.

   ...

   A quick overview of the approach we'll take for each phase: we'll disassemble
   the bomb with `objdump -d`, read the assembly to understand the logic, and
   use `gbb` to verify

   -- ... classic "initiation" hurdles... Don't worry... bomb isn't broken -- it;s
   just a matter of Linux permissions and how GDB communicates with the program. 

...

1. FIX "PERMISSION DENIED"
   The error `Permission denied`... means the file currently doesn't have 
   "execute" permissions. Even though you downloaded it, Linux needs to be 
   explicitly told that this file is a program it it allowed to run.

   THE FIX:
      Run this command in your terminal...

```Bash
chmod +x bomb
```
   `chmod +x` stands for "change mode + execute." This will allow the file to 
   run.


2. FIX "Undefined Command: `testing123`"
   You are trying to type your input directly into the `(gdb)` prompt. GBP
   thinks `testing123` is a command like `run` or `break`.

   THE CONCEPT:
      Inside GDB, there is a clear distinction between DEBUGGER COMMANDS (to
      control GDB) and PROGRAM INPUT (data you send to the bomb).

   THE CORRECT WORKFLOW:
      1. Start GDB: `gdb ./bomb`
      2. Set your safety net: `break explode_bomb`
      3. Set the phase start: `break phase_1`
      4. Run the program: Type `run` and hit Enter.
      5. Wait for the prompt: The bomb will print "Welcome to my fiendish little
         bomb..." and then pause. ONLY THEN do you type `testing123` and hit 
         Enter.      

---

   `objdump -d` is a command-line utility used to DISASSEMBLE a binary file. In
   the context..., it is the primary tool for turning the "unreadable" machine
   code (the `bomb` executable) back into human-readable assembly language.


WHAT IT DOES
   When Dr. Evil compiled the bomb, his C code was translated into a sequence of 
   bytes (machine code) that the CPU executes. Because you weren't given the 
   source code for the phases, `objdump -d` allows you to "reverse engineer" 
   the logic by showing you the assembly instructions those bytes represent.   


BREAKDOWN OF THE OUTPUT
   If you look at the screenshot you provided, or if you run the command 
   yourself, the output is typically organised into three columns:
      1. ADDRESS: The memory location where the instruction lives (e.g., `3f17`
         in your image).
      2. HEX DUMP (Opcodes): The actual bytes of the machine code (e.g., `55`,
         `48 89 e5`).
      3. INSTRUCTION: The human-readable assembly mnemonic and its operands 
         (e.g., `push %rbp` or `mov %rsp, %rbp`).


WHY IT IS USED IN THE BOMB LAB
   * UNDERSTANDING BEHAVIOR: By reading the assembly, you can see how the 
     program treats your input. For example, you can find where it compares your
     string to a "secret" string stored in memory.
   * LOCATING FUNCTIONS: You can use it to find the entry points for specific
     phases, such as `<phase_1>` or `<phase_2>`.
   * IDENTIFY SYSTEM CALLS: While `objdump -d` can sometimes by cryptic with
     system-level functions like `sscanf`, it provides the "skeleton" of the
     program's logic.     


HOW TO USE IT EFFECTIVELY
   To make the output easier to manage, student often redirect the results to a
   text file so they can search through it using a text editor ...

   `objdump -d bomb > assembly.txt`

      NOTE: The coursework specification mentions that while `objdump -d` is
      powerful, it doesn't tell the "whole story" for certain function calls. 
      For those, you'll need to use `gdb` to "disassemble within the debugger" 
      to see more detail. 

-- GNU/Linux is a FOSS, UNIX-like operating system composed of GNU software 
   tools (compilers, libraries, shell) and the Linux kernel, which manages 
   hardware resources. It enables users to run applications, manage files, and
   develop software, commonly distrbuted as packages like Ubuntu or Debian. It
   is often used for web servers, development, and desktop computing.

   
   KEY COMPONENTS AND USAGE
      * GNU Components: Provides essential userland tools, including the GCC
        compiler, Bash shell, and core utilities.
      * Linux Kernel: Manages system hardware, memory and processor resources,
      * Usage Examples: It powers nearly all top internet servers, powers 
        Android devices, runs desktop environments (GNOME, KDE), and is used in
        embedded systems.


  SYNONYMS AND NAMING
     * Commonly referred to as: Linux, Linux distribution, open-source OS.
     ...      


---

   ... hands dirty in the debugger is actually the most rewarding way to learn
   how the machine really thinks.

   Using `gdb` (the GNU Debugger) is highly recommended for this coursework
   because it lets you freeze time, inspect the CPU's exact state, and safely
   step through the bomb's logic without accidentally detonating it. Remember, 
   the specification warns that every time the bomb explodes, it sends a message
   to the lab server and you lose half a point. 

   Let's walk through the essential `gdb` workflow, step-by-step, to get you
   safely into Phase 1.


---
STEP 1: Launch GDB and Set Your Safety Nets   
   First, start the debugger with your bomb executable.

```Bash
gdb ./bomb
```

   Before you do anything else, you need to set breakpoints. A breakpoint tells
   `gdb` to pause execution right before it runs a specific function.

   Inside the `(gdb)` prompt, type:

```
(gdb) break explode_bomb
```

   This is your ultimate safety net. If you enter the wrong string, the bomb
   will try to call `explode_bomb`, but `gdb` will freeze it right before it
   executes, saving your grade!

   Next, set a breakpoint for the start of the first phase:

```
(gdb) break phase_1
```
   

---
STEP 2: Get the "Hacker" UI (TUI Mode)   
   Since you liked the layout your friend was using, let's turn that on. This is
   called the Text User Interface (TUI).

```
(gdb) layout asm
(gdb) layout regs
```

   This splits your terminal screen. The top window will show the CPU registers
   (like `%rax`, `%rdi`, etc.), the middle window will show the assembly code 
   highlighting exactly where you are, and the bottom window is your command 
   prompt.


---
STEP 3: Run the Program
   Now that your safety nets are up, start the program:

```
(gdb) run
```
   The program will start, print Dr Evil's welcome message, and then pause, 
   waiting for your input. Type in a dummy test string (e.g., `Testing123`) and
   hit enter.

   Because we set a breakpoint earlier, `gdb` will immediately pause the program 
   the moment it enters `<phase_1>` and give control back to you. 


---
STEP 4: Inspecting and Stepping (The Core Loop)
   You are now frozen inside Phase 1. Here are the three most important
   commands you will use to navigate and investigate:


1. DISASSEMBLE THE CURRENT FUNCTION:
   If you ever lose your place or aren't using the TUI mode, you can view the
   assembly for the function you are currently in.    
```
(gdb) disas
```

2. STEPPING FORWARD:
   To execute the assembly code one single line at a time, use "next 
   instruction":
```
(gdb) ni
```
   (Note: You can just press the `Enter` key on your keybord to repeat the
   last command, so you don't have to keep typing `ni`!)
   

3. INSPECTING MEMORY (Looking for the secret string)
   As you step through, you will likely see an instruction where an address is
   moved into a register right before a function call like `string_not_equal`.
   It might look like `mov $0x402400, %esi`.

   You can inspect what is actually stored at that memory address. To examine a
   string at a specific hex address, use:

```
(gdb) x/s 0x402400
```
   
   (Replace `0x402400` with whatever memory address you actually see in your 
   code). If you see a readable phrase pop up, congratulations, you've likely
   just found the password for that phrase!












---


   A quick overview of the approach we'll take for each phase: we'll disassemble
   the bomb with `objdump -d`, read the assembly to understand the logic, and
   use `gdb` to verify our understanding without detonating anything. I'll 
   explain the x86 concepts as they come up -- things like calling conventions,
   comparison instructions, conditional jumps, stack frames, and how C 
   consutrcts like loop,s arrays, and switch statements look in assembly.

   A couple of things to get us started. 
   In the emantime, a few quick tips...

   DON'T RUN THE BOMB WITHOUT GDB. Always launch it as `gdb ./bomb` and set a
   breakpoint on `explode_bomb` first (``)